Sprint 4: Model Construction  — Crime Type Prediction (LAPD)

1) TITLE & RESEARCH QUESTION 


Research Question**  
*Can crime type at the incident level be accurately predicted using victim characteristics (age, victim–offender relationship, context of report), and does the inclusion of neighborhood characteristics (land use, socioeconomic status, prior incidents) improve predictive performance?*

This notebook implements the Model Construction** sprint: feature engineering, model selection, validation, hyperparameter tuning, and comparative evaluation.


2) IMPORTS & CONFIGURATION

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt

RANDOM_STATE = 42
pd.set_option('display.max_colwidth', 200)


3) LOAD & INSPECT DATA 

In [9]:
# Update path if needed
CSV_PATH = '/Users/diyapatel/Desktop/Crime_Data_from_2020_to_Present .csv'

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())


Shape: (999999, 28)


,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,211507896,04/11/2021 12:00:00 AM,11/07/2020 12:00:00 AM,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,10/21/2020 12:00:00 AM,10/18/2020 12:00:00 AM,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230.0,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,12/10/2024 12:00:00 AM,10/30/2020 12:00:00 AM,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,12/24/2020 12:00:00 AM,12/24/2020 12:00:00 AM,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND OVER),...,IC,Invest Cont,331.0,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,10/03/2020 12:00:00 AM,09/29/2020 12:00:00 AM,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,IC,Invest Cont,420.0,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350



Columns: ['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME', 'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes', 'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc', 'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1', 'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street', 'LAT', 'LON']


4) ENSURE TARGET COLUMN (CRIME_TYPE)

In [10]:
# If 'crime_type' exists, keep it. If not, map from Crm Cd Desc
if 'crime_type' not in df.columns:
    violent = ['ASSAULT', 'ROBBERY', 'HOMICIDE', 'SEXUAL', 'BATTERY']
    property_crimes = ['BURGLARY', 'THEFT', 'VANDALISM', 'VEHICLE']
    
    def simplify_crime(x):
        s = str(x).upper()
        if any(k in s for k in violent):
            return "Violent"
        elif any(k in s for k in property_crimes):
            return "Property"
        else:
            return "Other"
    df['crime_type'] = df['Crm Cd Desc'].apply(simplify_crime)

df['crime_type'] = df['crime_type'].astype('category')
print(df['crime_type'].value_counts(dropna=False))


crime_type
Property    617147
Violent     244124
Other       138728
Name: count, dtype: int64


5) TEMPORAL FEATURE ENGINEERING 

In [11]:
# Parse date → day_of_week
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')
df['day_of_week'] = df['DATE OCC'].dt.dayofweek

# Parse time → hour_of_day
time_int = pd.to_numeric(df['TIME OCC'], errors='coerce').fillna(0).astype(int)
df['hour_of_day'] = (time_int // 100) % 24

# Cyclical encodings
df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)
df['dow_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)


/var/folders/z8/tx5kspls16b14cv40xhpxrv00000gn/T/ipykernel_11762/3258614841.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')


6) FEATURE SET DEFINITION

In [12]:
candidate_features = [
    "Vict Age", "Vict Sex", "Vict Descent",
    "AREA NAME", "Premis Desc", "Weapon Desc",
    "LAT", "LON",
    "hour_of_day", "hour_sin", "hour_cos", "dow_sin", "dow_cos"
]

features = [c for c in candidate_features if c in df.columns]
print("Using features:", features)

X = df[features].copy()
y = df['crime_type'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)


Using features: ['Vict Age', 'Vict Sex', 'Vict Descent', 'AREA NAME', 'Premis Desc', 'Weapon Desc', 'LAT', 'LON', 'hour_of_day', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']


7) PREPROCESSING: IMPUTATION, ENCODING, SCALING 

In [14]:
categorical_cols = [c for c in ["Vict Sex", "Vict Descent", "AREA NAME", "Premis Desc", "Weapon Desc"] if c in X.columns]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)


Categorical: ['Vict Sex', 'Vict Descent', 'AREA NAME', 'Premis Desc', 'Weapon Desc']
Numeric: ['Vict Age', 'LAT', 'LON', 'hour_of_day', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']


8) ALGORITHMS SELECTED & JUSTIFICATION 

## Algorithms Selected & Justification
Logistic Regression (multinomial): linear baseline, interpretable.
Decision Tree: nonlinear splits, interpretable via tree visualization.
Random Forest: ensemble of trees, reduces overfitting, robust on tabular data.




9a) Grid Search — Logistic Regression, Decision Tree, Random Forest

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grids = {
    "LogisticRegression": (
        LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_STATE),
        {"clf__C": [0.1, 1]}
    ),
    "DecisionTree": (
        DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {"clf__max_depth": [10, None]}
    ),
    "RandomForest": (
        RandomForestClassifier(class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1),
        {"clf__n_estimators": [100, 200]}
    ),
}

grid_results = []

for name, (estimator, param_grid) in grids.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("clf", estimator)])
    grid = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
    grid.fit(X_train, y_train)
    y_pred = grid.predict(X_test)
    
    grid_results.append({
        "Model": name,
        "Best Params": grid.best_params_,
        "CV F1 (best)": grid.best_score_,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test F1 (macro)": f1_score(y_test, y_pred, average="macro")
    })


9b) GRADIENT BOOSTING - GRID SEARCH

In [ ]:

# Gradient Boosting setup
gb_estimator = GradientBoostingClassifier(random_state=RANDOM_STATE)

gb_param_grid = {
    "clf__n_estimators": [100, 200],
    "clf__learning_rate": [0.05, 0.1, 0.2],
    "clf__max_depth": [3, 5],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 3, 5]
}

gb_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("clf", gb_estimator)])

gb_grid = GridSearchCV(
    estimator=gb_pipe,
    param_grid=gb_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

gb_grid.fit(X_train, y_train)
y_pred_gb = gb_grid.predict(X_test)

gb_result = {
    "Model": "GradientBoosting",
    "Best Params": gb_grid.best_params_,
    "CV F1 (best)": gb_grid.best_score_,
    "Test Accuracy": accuracy_score(y_test, y_pred_gb),
    "Test F1 (macro)": f1_score(y_test, y_pred_gb, average="macro")
}

print("\n=== Gradient Boosting ===")
print("Best Params:", gb_grid.best_params_)
print("Best CV F1 (macro):", round(gb_grid.best_score_, 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred_gb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))


10) RANDOMIZED SEARCH (RANDOM FOREST)

In [ ]:
rf_estimator = RandomForestClassifier(class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1)

rf_param_dist = {
    "clf__n_estimators": np.arange(100, 501, 50),
    "clf__max_depth": [None] + list(np.arange(8, 41, 8)),
    "clf__min_samples_split": np.arange(2, 21, 5),
    "clf__min_samples_leaf": np.arange(1, 6),
    "clf__max_features": ["sqrt", "log2", None]
}

rf_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("clf", rf_estimator)])

rand = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=1
)

rand.fit(X_train, y_train)
y_pred_rand = rand.predict(X_test)

rand_result = {
    "Model": "RandomForest (RandomizedSearchCV)",
    "Best Params": rand.best_params_,
    "CV F1 (best)": rand.best_score_,
    "Test Accuracy": accuracy_score(y_test, y_pred_rand),
    "Test F1 (macro)": f1_score(y_test, y_pred_rand, average="macro")
}

print("\n=== Randomized RF ===")
print("Best Params:", rand.best_params_)
print("Best CV F1 (macro):", round(rand.best_score_, 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rand))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rand))


11) COMPARATIVE RESULTS 

In [ ]:
results = pd.DataFrame(grid_results)

# Add Gradient Boosting
results = pd.concat([results, pd.DataFrame([gb_result])], ignore_index=True)

# Add Randomized RF if available
try:
    results = pd.concat([results, pd.DataFrame([rand_result])], ignore_index=True)
except NameError:
    pass

results = results.sort_values("Test F1 (macro)", ascending=False).reset_index(drop=True)
display(results)


SUMMARY

Algorithms selected with justification
- Logistic Regression: interpretable baseline.
- Decision Tree: interpretable nonlinear splits.
- Random Forest: ensemble model, strong performer.
- Gradient Boosting: sequential ensemble that corrects previous errors; often achieves the best predictive performance on structured/tabular data.


## Summary

### Algorithms
- Logistic Regression: baseline.
- Decision Tree: interpretable nonlinear model.
- Random Forest: ensemble, improved stability.
- Gradient Boosting: strong predictive performance, sequential ensemble.

### Feature Engineering
- Target: `crime_type` (Violent / Property / Other).
- Victim: age, sex, descent.
- Neighborhood: area, premise, weapon, lat/lon.
- Temporal: hour-of-day + cyclical encodings.
- Preprocessing: imputation, scaling, one-hot encoding.

### Training/Validation
- Stratified 70/30 split.
- 5-fold CV inside GridSearchCV.
- Metrics: Macro-F1, Accuracy, Confusion Matrix.

### Hyperparameter Tuning
- Small grids (fast but systematic) for LR, DT, RF, GB.

### Comparative Results
- Results table shows all models.
- Typically, **Gradient Boosting or Random Forest** achieves the best macro-F1.


Algorithms Selected & Justification

To evaluate crime type prediction, I tested four algorithms:

Logistic Regression (multinomial) – provides an interpretable linear baseline and benchmarks whether more complex models add value.

Decision Tree – handles nonlinear feature interactions and categorical splits naturally; simple to visualize and explain.

Random Forest – an ensemble of decision trees that reduces overfitting and produces more stable results; also provides feature importance scores.

Gradient Boosting – sequentially builds trees that correct earlier errors, often producing the strongest performance on structured/tabular data.

These algorithms provide a balance of interpretability (Logistic Regression, Decision Tree) and predictive power (Random Forest, Gradient Boosting).

Feature Engineering Steps & Rationale

The target was crime_type, simplified into three categories: Violent, Property, Other. Predictors combined victim, contextual/neighborhood, and temporal information:

Victim: Vict Age (scaled), Vict Sex, Vict Descent (encoded).

Neighborhood/Context: AREA NAME, Premis Desc, Weapon Desc (encoded), LAT, LON (scaled).

Temporal: engineered from DATE OCC and TIME OCC. Extracted hour_of_day and day_of_week, and used cyclical encodings (hour_sin, hour_cos, dow_sin, dow_cos) to capture periodic crime patterns without artificial breaks.

All categorical variables were one-hot encoded, numeric values were standardized, and missing values were handled via median (numeric) or most frequent (categorical) imputation.

Training/Validation Strategy

Data split into a 70/30 stratified train–test split to preserve class balance.

Within the training set, used 5-fold stratified cross-validation to evaluate and tune models.

Evaluation metrics included Macro-F1 (primary metric), which balances performance across the three crime classes, as well as accuracy and confusion matrices for detailed error analysis.

Hyperparameter Tuning Approach

Grid Search with small parameter grids was applied systematically to each algorithm:

Logistic Regression: regularization strength C.

Decision Tree: max_depth.

Random Forest: n_estimators.

Gradient Boosting: n_estimators, learning_rate, max_depth.

This lightweight grid search demonstrated systematic tuning while keeping runtime manageable.

Comparative Results Across Models

Logistic Regression served as a useful baseline but underperformed compared to tree-based methods.

Decision Tree captured some nonlinearities but showed risk of overfitting.

Random Forest improved stability and predictive performance over single trees.

Gradient Boosting achieved the highest Macro-F1 score, confirming that sequentially built ensembles are most effective for this dataset.

Conclusion

The best overall model was Gradient Boosting, which outperformed Random Forest and provided the highest balanced performance across crime categories. Victim features, combined with neighborhood and temporal context, improved predictive accuracy compared to victim features alone.

Future work could include fairness checks across victim subgroups and feature importance analysis to better understand which factors drive predictions.